# VocabOps — GPU Draft Curation (Colab Runner)

Thin reproducible runner for `python -m vocabops.draft_curation` (see [Issue #23](https://github.com/hamakyo/vocabops/issues/23)).

Generates unreviewed Japanese general & software-engineering glosses, CEFR estimates, allowlisted categories, synthetic corpus-grounded example sentences (EN + JA), and deterministic cloze cards using an open-weights GPU LLM.

**Guardrails**
- All prompting, parsing, cloze masking, and schema assembly live in `src/vocabops/draft_curation.py`; this notebook only installs the repo and runs the CLI.
- Every output record is tagged `curation.status = "draft"`, `needs_human_review = true`, and `examples[].kind = "synthetic"`.
- Uses greedy decoding and a fixed `SNAPSHOT_ID` for deterministic outputs.

In [ ]:
#@title 1. Run parameters
REPO_URL = "https://github.com/hamakyo/vocabops.git" #@param {type:"string"}
GIT_REF = "main" #@param {type:"string"}
USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
DRIVE_WORKDIR = "/content/drive/MyDrive/vocabops" #@param {type:"string"}
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct" #@param ["Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]
ENRICHMENT_PATH = "data/core500-vocab-v3-enrichment.jsonl" #@param {type:"string"}
RANKING_PATH = "data/core500-vocab-v3.jsonl" #@param {type:"string"}
SNAPSHOT_ID = "core500-draft-v1" #@param {type:"string"}
METRIC_SNAPSHOT = "combined-3source-enrich-v1" #@param {type:"string"}
MIN_POS_SHARE = 0.25 #@param {type:"number"}
BATCH_SIZE = 16 #@param {type:"integer"}
LIMIT = 0 #@param {type:"integer"}

import subprocess

gpu_check = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu_check.returncode == 0:
    print("GPU detected:", gpu_check.stdout.strip())
else:
    print("No GPU detected; a GPU runtime (T4 or L4) is strongly recommended.")


In [ ]:
#@title 2. Mount Drive (optional), clone repo, and install .[draft]
import os
import subprocess
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path(DRIVE_WORKDIR)
else:
    WORK_ROOT = Path("/content/work")

WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path("/content/vocabops")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--prune"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", GIT_REF], check=False)

os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "-e", ".[draft]"], check=True)

print("Checked out:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("Work root:  ", WORK_ROOT)


In [ ]:
#@title 3. Run python -m vocabops.draft_curation
import subprocess

OUTPUT_DIR = WORK_ROOT / "out" / SNAPSHOT_ID
cmd = [
    "python",
    "-m",
    "vocabops.draft_curation",
    "--enrichment",
    ENRICHMENT_PATH,
    "--output",
    str(OUTPUT_DIR),
    "--snapshot-id",
    SNAPSHOT_ID,
    "--metric-snapshot",
    METRIC_SNAPSHOT,
    "--model",
    MODEL_NAME,
    "--min-pos-share",
    str(MIN_POS_SHARE),
    "--batch-size",
    str(BATCH_SIZE),
]
if RANKING_PATH.strip():
    cmd.extend(["--ranking", RANKING_PATH.strip()])
if LIMIT > 0:
    cmd.extend(["--limit", str(LIMIT)])

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
#@title 4. Inspect manifest.json and sample draft records
import json

manifest_path = OUTPUT_DIR / "manifest.json"
drafts_path = OUTPUT_DIR / "drafts.jsonl"

print("=== manifest.json ===")
print(manifest_path.read_text(encoding="utf-8"))

print("=== first 10 draft records ===")
with drafts_path.open("r", encoding="utf-8") as handle:
    for idx, line in enumerate(handle):
        if idx >= 10:
            break
        rec = json.loads(line)
        ex = rec["examples"][0]["text"] if rec["examples"] else ""
        print(
            f"{rec['id']:<20} ({rec['cefr']}) {rec['ja']['gloss']} / {rec['ja']['software_gloss']}\n"
            f"  ex: {ex}\n"
            f"  warnings: {rec['curation']['validation_warnings']}"
        )
